# Listen to Segment Cliques

Using this notebook you can listen to the segment cliques.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import json
import math
import csv
import json
from pathlib import Path
from collections import defaultdict
import random

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from IPython.display import Audio, display, YouTubeVideo, HTML

In [2]:
def yt_iframe(video_id, start=0, width=400, height=225):
    return (f'<iframe width="{width}" height="{height}" '
            f'src="https://www.youtube.com/embed/{video_id}?start={start}" '
            f'frameborder="0" allowfullscreen></iframe>')

def display_row_yt(row):
    yt_id = row['youtube_id'].item()
    start = int(round(float(row["segment_start_time"].item())))
    display(HTML(f"""
    <div style="display:flex; gap:12px; margin:8px 0;">
      <div>{yt_iframe(yt_id, start)}</div>
    </div>
    """))

## Load the segment clique metadata

In [3]:
# This csv is small enough to fit to memory
csv_path = Path(
    "/projects/mtg/projects/unified-similarity/dvi-similar-regions/Discogs-VI-SIREN/train/segment-cliques.csv"
)
df = pd.read_csv(csv_path)
df = df.sort_values(['track_clique_id', 'segment_clique_id', 'version_id'])
print(f"{len(df):,}")
display(df.head())

18,566,717


,track_clique_id,segment_clique_id,version_id,youtube_id,segment_start_time,segment_end_time,track_duration
17567298,C-0000000,S-0492995,V-0000004,MI23duPUNaY,19.0,39.0,216.399994
17567300,C-0000000,S-0492995,V-0000004,MI23duPUNaY,46.0,66.0,216.399994
17567301,C-0000000,S-0492995,V-0000004,MI23duPUNaY,85.0,105.0,216.399994
17567303,C-0000000,S-0492995,V-0000004,MI23duPUNaY,93.0,113.0,216.399994
17567305,C-0000000,S-0492995,V-0000004,MI23duPUNaY,10.0,30.0,216.399994


## Sample and Listen

### Sample a random segment clique

In [4]:
random_segment_clique_id = df['segment_clique_id'].sample(1).iloc[0]
# random_segment_clique_id = "S-0007601"

segment_clique = df[df['segment_clique_id'] == random_segment_clique_id]
print(f"The segment clique is made up of {segment_clique['version_id'].nunique()} unique versions.")
print(f"There are {len(segment_clique):,} segments in the segment clique.")

display(segment_clique.sample(frac=1).head(n=min(10, len(segment_clique))))

The segment clique is made up of 24 unique versions.
There are 1,898 segments in the segment clique.


,track_clique_id,segment_clique_id,version_id,youtube_id,segment_start_time,segment_end_time,track_duration
6180184,C-0040288,S-0016428,V-0371479,kWVR56oXwgM,26.0,46.0,157.600006
6179693,C-0040288,S-0016428,V-0371446,kJ5lAuqV0aM,21.0,41.0,144.399994
6179825,C-0040288,S-0016428,V-0371480,jK8rwwjL7E4,19.0,39.0,189.000000
6180261,C-0040288,S-0016428,V-0371444,50AJA4gvGRI,39.0,59.0,140.899994
6179839,C-0040288,S-0016428,V-0371482,9w67vs5H5ZE,47.0,67.0,193.100006
6179347,C-0040288,S-0016428,V-0371479,kWVR56oXwgM,130.0,150.0,157.600006
6180691,C-0040288,S-0016428,V-0371485,Ib-7NCI2iKs,33.0,53.0,135.600006
6180705,C-0040288,S-0016428,V-0371457,eq1e4gI0koc,64.0,84.0,156.100006
6180638,C-0040288,S-0016428,V-0371482,9w67vs5H5ZE,119.0,139.0,193.100006
6179614,C-0040288,S-0016428,V-0371479,kWVR56oXwgM,36.0,56.0,157.600006


#### Sample N random versions and 1 segment per version

In [5]:
one_per_version = segment_clique.groupby('version_id').sample(1)
for i in range(min(10, len(one_per_version))):
    display_row_yt(one_per_version.iloc[[i]])

In [6]:
N = 62

version_ids = segment_clique['version_id'].unique()
n = min(N, len(version_ids))
sampled_versions = np.random.choice(version_ids, size=n, replace=False)

result = segment_clique[
    segment_clique['version_id'].isin(sampled_versions)
].groupby('version_id').sample(1)

for i in range(len(result)):
    print(result.iloc[i].youtube_id)
#     display_row(result.iloc[[i]])
    display_row_yt(result.iloc[[i]])

E-CN7JCnztg


Qe1v0ndNQ0M


0Ji_ffRzvRQ


PHZl4c5nn1w


A1fnETwVlQM


...

jK8rwwjL7E4


9w67vs5H5ZE


oj5KwScyCZU


Ib-7NCI2iKs


XiH9rupwZ84


#### N segments from a random version

In [8]:
random_version_id = segment_clique['version_id'].sample(1).iloc[0]
random_version = segment_clique[segment_clique['version_id'] == random_version_id]

for i in range(min(N, len(random_version))):
    display_row_yt(random_version.iloc[[i]])

#### Random N Segments

In [9]:
random_version_id = segment_clique['version_id'].sample(1).iloc[0]
print(random_version_id)
random_version = segment_clique[
    segment_clique['version_id'] == random_version_id
].sample(frac=1)

for i in range(min(N, len(random_version))):
    display_row_yt(random_version.iloc[[i]])

V-0371480


### Sample from size M track cliques

Here we can display the segment cliques that are derived from the same track clique.

In [15]:
mapping = df.groupby("track_clique_id")["version_id"].unique()   # Series of ndarrays
mapping = mapping.to_dict()                                      # dict[str, ndarray]
# mapping = {k: len(v) for k,v in mapping.items()}

In [19]:
M = 3

clique_ids_size_M = list(k for k,v in mapping.items() if len(v)==M)
print(f'{len(clique_ids_size_M):,}')

12,139


In [20]:
clique_id = random.choice(clique_ids_size_M)
track_clique = df[df['track_clique_id'] == clique_id]
display(track_clique.head(n=10))

,track_clique_id,segment_clique_id,version_id,youtube_id,segment_start_time,segment_end_time,track_duration
16153601,C-0134743,S-0264980,V-1204331,p5Fi4UYkN7I,68.0,88.0,180.399994
16153603,C-0134743,S-0264980,V-1204331,p5Fi4UYkN7I,69.0,89.0,180.399994
16153602,C-0134743,S-0264980,V-1204334,NNveMYGG7SM,7.0,27.0,161.699997
16153604,C-0134743,S-0264980,V-1204337,jWf5vKBwEO0,13.0,33.0,158.899994
16153605,C-0134743,S-0264980,V-1204337,jWf5vKBwEO0,67.0,87.0,158.899994
16153606,C-0134743,S-0264981,V-1204331,p5Fi4UYkN7I,18.0,38.0,180.399994
16153607,C-0134743,S-0264981,V-1204334,NNveMYGG7SM,17.0,37.0,161.699997
16153609,C-0134743,S-0264981,V-1204334,NNveMYGG7SM,71.0,91.0,161.699997
16153608,C-0134743,S-0264981,V-1204337,jWf5vKBwEO0,72.0,92.0,158.899994
16153610,C-0134743,S-0264981,V-1204337,jWf5vKBwEO0,18.0,38.0,158.899994


In [21]:
for seg_clique_id, segment_clique in track_clique.groupby('segment_clique_id'):
    print(f"\nsegment_clique_id: {seg_clique_id}")
    display(segment_clique.head(n=10))
    for i in range(min(10, len(segment_clique))):
        display_row_yt(segment_clique.iloc[[i]])


segment_clique_id: S-0264980


,track_clique_id,segment_clique_id,version_id,youtube_id,segment_start_time,segment_end_time,track_duration
16153601,C-0134743,S-0264980,V-1204331,p5Fi4UYkN7I,68.0,88.0,180.399994
16153603,C-0134743,S-0264980,V-1204331,p5Fi4UYkN7I,69.0,89.0,180.399994
16153602,C-0134743,S-0264980,V-1204334,NNveMYGG7SM,7.0,27.0,161.699997
16153604,C-0134743,S-0264980,V-1204337,jWf5vKBwEO0,13.0,33.0,158.899994
16153605,C-0134743,S-0264980,V-1204337,jWf5vKBwEO0,67.0,87.0,158.899994



segment_clique_id: S-0264981


,track_clique_id,segment_clique_id,version_id,youtube_id,segment_start_time,segment_end_time,track_duration
16153606,C-0134743,S-0264981,V-1204331,p5Fi4UYkN7I,18.0,38.0,180.399994
16153607,C-0134743,S-0264981,V-1204334,NNveMYGG7SM,17.0,37.0,161.699997
16153609,C-0134743,S-0264981,V-1204334,NNveMYGG7SM,71.0,91.0,161.699997
16153608,C-0134743,S-0264981,V-1204337,jWf5vKBwEO0,72.0,92.0,158.899994
16153610,C-0134743,S-0264981,V-1204337,jWf5vKBwEO0,18.0,38.0,158.899994



segment_clique_id: S-0264982


...

,track_clique_id,segment_clique_id,version_id,youtube_id,segment_start_time,segment_end_time,track_duration
16153682,C-0134743,S-0264993,V-1204334,NNveMYGG7SM,57.0,77.0,161.699997
16153683,C-0134743,S-0264993,V-1204337,jWf5vKBwEO0,59.0,79.0,158.899994



segment_clique_id: S-0264994


,track_clique_id,segment_clique_id,version_id,youtube_id,segment_start_time,segment_end_time,track_duration
16153684,C-0134743,S-0264994,V-1204334,NNveMYGG7SM,4.0,24.0,161.699997
16153685,C-0134743,S-0264994,V-1204337,jWf5vKBwEO0,7.0,27.0,158.899994
16153686,C-0134743,S-0264994,V-1204337,jWf5vKBwEO0,64.0,84.0,158.899994



segment_clique_id: S-0264995


,track_clique_id,segment_clique_id,version_id,youtube_id,segment_start_time,segment_end_time,track_duration
16153687,C-0134743,S-0264995,V-1204334,NNveMYGG7SM,90.0,110.0,161.699997
16153688,C-0134743,S-0264995,V-1204337,jWf5vKBwEO0,38.0,58.0,158.899994


### Sample from segment cliques with M unique versions

In [22]:
M = 2
seg_clique_size_M_versions = df.groupby('segment_clique_id').filter(lambda x: x['version_id'].nunique() == M)

In [23]:
random_seg_clique_id = random.choice(seg_clique_size_M_versions['segment_clique_id'].unique())

segment_clique = df[df['segment_clique_id'] == random_seg_clique_id]
display(segment_clique)

for i in range(len(segment_clique)):
    display_row_yt(segment_clique.iloc[[i]])

,track_clique_id,segment_clique_id,version_id,youtube_id,segment_start_time,segment_end_time,track_duration
16420130,C-0072818,S-0302532,V-0753019,QCo-c_Zpd9w,4.0,24.0,388.799988
16420131,C-0072818,S-0302532,V-0753020,V8HUZiEWF1s,333.0,353.0,381.200012


### Sample from segment cliques of size M

In [24]:
M = 2
seg_clique_size_M_segments = df.groupby('segment_clique_id').filter(lambda x: len(x) == M)

In [25]:
random_seg_clique_id = random.choice(seg_clique_size_M_segments['segment_clique_id'].unique())

segment_clique = df[df['segment_clique_id'] == random_seg_clique_id]
display(segment_clique)

for i in range(len(segment_clique)):
    display_row_yt(segment_clique.iloc[[i]])

,track_clique_id,segment_clique_id,version_id,youtube_id,segment_start_time,segment_end_time,track_duration
17474317,C-0140304,S-0472742,V-1232429,jDGgFQmB9v4,20.0,40.0,162.800003
17474318,C-0140304,S-0472742,V-1232430,B6QL4q_h-1I,114.0,134.0,199.699997


## Duplicate Finder

This simple rule can be used as a duplicate finder

In [ ]:
zero_diff = seg_clique_size_M_segments.groupby('segment_clique_id').filter(
    lambda group: abs(group['segment_end_time'].diff().iloc[-1]) == 0
)

In [ ]:
display(zero_diff)

In [ ]:
random_seg_clique_id = random.choice(zero_diff['segment_clique_id'].unique())

segment_clique = df[df['segment_clique_id'] == random_seg_clique_id]
display(segment_clique)

for i in range(len(segment_clique)):
    display_row(segment_clique.iloc[[i]])

## If you have the audio files

In [ ]:
# current_dir = Path.cwd()
# parent_dir = current_dir.parent
# if str(parent_dir) not in sys.path:
#     sys.path.append(str(parent_dir))
# print(f"Added {parent_dir} to sys.path")

# from src.common.audio import load_audio, SAMPLE_RATE

# music_dir = Path("/projects/mtg/projects/unified-similarity/datasets/discogs-vi-yt-16kHz/audio/")
# cliques_json_path = Path(
#     "/projects/mtg/projects/version-identification/datasets/Discogs-VI/dataset_release/discogs_20240701/main/Discogs-VI-YT-20240701-light.json.train"
# )

# def display_row(row):

#     yt_id = row['youtube_id'].item()

#     audio_path = music_dir / yt_id[:2] / f'{yt_id}.wav'

#     t0 = row["segment_start_time"].item()
#     t1 = row["segment_end_time"].item()

#     audio = load_audio(
#         audio_path, start=int(t0*SAMPLE_RATE), length=int((t1-t0)*SAMPLE_RATE)
#     )

#     display(Audio(audio, rate=SAMPLE_RATE, normalize=False))    